In [ ]:
working_directory = "C:/Users/Kaist/Desktop/EE305_module2" # working director, change this if you use your PC
# change current directory to the working directory
%cd {working_directory}

In [ ]:
import sys
print(sys.version) # print Python version
print(sys.executable) # print path of the Python executable

In [ ]:
import paramiko # library for SSH communication
import numpy # numerical computing library
import matplotlib # library for plotting and visualization
import time # library for time-related function

print("All imports successful")

In [ ]:
# PyQt6 : GUI framework (used for real-time plotting with matplotlib/pyqtgraph)
!pip install PyQt6 

In [ ]:
HOST = "ee30005.local" # raspberry PI's hostname
USER = "student" # user name
PW   = "student" # password

ssh = paramiko.SSHClient() # create SSH client instance

ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy()) # skip host key verification

# try to establish SSH connection and execute command
try:
    ssh.connect(HOST, username=USER, password=PW, timeout=10) # connect to raspberrp PI using SSH credentials (10s timeout)
    print("SSH connection success!")

    stdin, stdout, stderr = ssh.exec_command("hostname") # execute 'hostname' command on raspberry PI
    print("Remote Hostname:", stdout.read().decode().strip())

except Exception as e:
    print("SSH connection fail:", e) # if there is an error, print log

finally:
    ssh.close() # ensure SSH connection is closed

In [ ]:
# SSH connection to raspberry PI
def ssh_connect():
    ssh = paramiko.SSHClient() # create new SSH client
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy()) # skip verification
    ssh.connect(HOST, username=USER, password=PW, timeout=10) # connect
    return ssh

# execute command in raspberry PI
def ssh_run(ssh, cmd, timeout=120):
    stdin, stdout, stderr = ssh.exec_command(cmd, timeout=timeout) # excute command, capture input/output/error streams
    out = stdout.read().decode(errors="replace") # read stdout, decode bytes to string (replace invalid chars)
    err = stderr.read().decode(errors="replace") # read stderr, decode bytes to string (replace invalid chars)
    return out, err

In [ ]:
cmd = f"arecord -l" # command to list available audio recording devices (ALSA)

ssh = ssh_connect() # ssh connection
out, err = ssh_run(ssh, cmd, timeout=10) # execute arecord -l command (10s timeout), capture output/error
print("---- arecord stdout ----")
print(out)  # print captured hardware devices list from stdout
print("---- arecord stderr ----")
print(err) # print stderr (usually empty for this command)

In [ ]:
SAMPLING_RATE = 48000 # sampling rate : 48000 Hz
CHANNEL = 1 # channel : 1
DEVICE = "plughw:0,0" # If you have a problem, change it to "default"
RECORDING_DURATION = 20 # recording duration : 20 seconds
DATATYPE = "raw" # get raw PCM data (not including .wav header)
BUFFER_SIZE = 4096 # buffer size
PERIOD_SIZE = 1024 # period size, when the buffer accumulates one period of samples, ALSA signals that the data is ready to be transferred

cmd = f"arecord -D {DEVICE} -f S32_LE -r {SAMPLING_RATE} -c {CHANNEL} -d {RECORDING_DURATION} -t {DATATYPE} --buffer-size {BUFFER_SIZE} --period-size {PERIOD_SIZE}" # recording command

In [ ]:
import numpy as np
def remove_dc_offset(x): 
    return x - np.mean(x) # Remove dc offset

def compute_powerspec_db(x, chunk_size, window, fft_size):
    if len(x) < chunk_size: 
        x = np.concatenate((x, np.zeros(chunk_size - len(x), dtype=x.dtype)))  # Zero-pad the signal if it is shorter than the chunk size
    xw = x * window # Applying the window
    xw = xw/np.linalg.norm(window)*np.sqrt(len(window)) # Compensate for energy loss due to windowing
    
    X = np.fft.rfft(xw, n=fft_size) # Compute the real FFT of the windowed signal
    mag = np.abs(X) / chunk_size # Scale magnitude by signal length
    power = mag**2 # power
    if fft_size % 2 == 0: # Convert to single-sided spectrum (preserve DC and Nyquist if present)  
        power[1:-1] *= 2.0 # Even FFT size → includes Nyquist bin
    else:
        power[1:] *= 2.0 # Odd FFT size → no Nyquist bin
    power_db = 10 * np.log10(power) # Convert power spectrum to decibel scale (dB)
    return power_db

In [ ]:
## Do not stop the cell while the code is running.
# If no audio device is found, or if only a busy audio device is found after stopping the code,
# reboot the Raspberry Pi.

# Use Qt backend for real-time plotting (separate GUI window)
%matplotlib qt
import time 
import numpy as np
import matplotlib.pyplot as plt

# constants for receiving data & processing data
CHUNK_SIZE = 2048         # Number of samples per processing 
# CHUNK_SIZE = 4096        
WINDOW = np.hanning(CHUNK_SIZE).astype(np.float32) # Hann window for spectral analysis
# WINDOW = np.ones(CHUNK_SIZE).astype(np.float32) # Rect window
# FFT_SIZE= 2048 # Number of FFT points
FFT_SIZE = 4096  
HOP_SIZE = CHUNK_SIZE // 2 # Hop size 

BYTES_PER_SAMPLE = 4      # 4 bytes per sample for S32_LE format
CHUNK_BYTES = CHUNK_SIZE * BYTES_PER_SAMPLE # Number of bytes per processing block
HOP_BYTES = HOP_SIZE * BYTES_PER_SAMPLE # Number of bytes for hop
RECV_SIZE = CHUNK_BYTES * 8  # Maximum number of bytes to read from the SSH channel at once
MAX_BACKLOG_CHUNKS = 10 # Maximum number of chunks allowed in the backlog
MAX_BACKLOG_BYTES = CHUNK_BYTES * MAX_BACKLOG_CHUNKS # Maximum backlog size in bytes

# constants for plotting
FPS = 20                # Plot update rate in frames per second
PLOT_INTERVAL = 1.0 / FPS   # Minimum time interval between plot updates
YMAX = -30 # maximum y-axis value
YMIN = -130 # minimum y-axis value

# figure initialization
plt.ion() 
freqs = np.fft.rfftfreq(FFT_SIZE, d=1.0/SAMPLING_RATE)  # Frequency axis for the FFT
valid_freqs = freqs >= 20  # Keep frequencies above 20 Hz for log-scale display
freqs = freqs[valid_freqs] # frequency axis above 20 Hz
dummy = np.full_like(freqs, -120.0) # Dummy spectrum for initial plot display

fig, ax = plt.subplots(figsize=(10, 5))
line, = ax.plot(freqs, dummy, animated=True)
ax.set_title(f"Real-time Power Spectrum (Mono, plughw), CHUNK_SIZE={CHUNK_SIZE}, FFT_SIZE={FFT_SIZE}") # title
ax.set_xlabel("Frequency [Hz]") # x-axis label
ax.set_ylabel("Power [dB]") # y-axis label
ax.set_xscale("log") # x-axis - log scale
ax.set_xlim(20, SAMPLING_RATE / 2) # x-axis limit (20Hz ~ Nyquist frequency)
ax.set_ylim(YMIN, YMAX) # y-axis limit (dB), **change this if you want to change the y-axis range**
ax.grid(True, which="both", alpha=0.3) # grid

fig.canvas.draw() # Draw the full figure once to initialize the canvas
background = fig.canvas.copy_from_bbox(ax.bbox) # Save the static background of the axes for fast blitting updates
ax.draw_artist(line) # Draw the line artist once on top of the saved background
fig.canvas.blit(ax.bbox) # Update only the axes region for efficient real-time rendering
fig.canvas.flush_events() # Process GUI events so the initial frame appears


# ssh connection
ssh = ssh_connect() # Create SSH connection
stdin, stdout, stderr = ssh.exec_command(cmd) # Execute the remote recording command
channel = stdout.channel # SSH channel used to receive raw PCM bytes

# Buffers and counters
buffer_pcm = bytearray() # Buffer for PCM data
chunk_count = 0 # Number of processed chunks
dropped_chunks_count = 0 # Number of dropped chunks due to backlog overflow

# Real-time spectrum loop
time.sleep(0.1) # wait for plot initialization
print("Starting : real time  spectrum...")
start_time = time.time() # start time
last_plot_time = 0.0 # last plot time

while True:
    # ** Stop slightly before the nominal recording duration to avoid potential GUI (figure window) issue **
    if (time.time() - start_time) > (RECORDING_DURATION - 0.2): # ** higly recommended **
        break

    if channel.recv_ready():
        data = channel.recv(RECV_SIZE) # Read up to RECV_SIZE(maximum) bytes from the SSH channel
        buffer_pcm.extend(data) # Append received bytes to the local PCM buffer

        # If the backlog grows too large, drop the oldest chunks (to keep the display close to real time)
        if len(buffer_pcm) > MAX_BACKLOG_BYTES:
            overflow_bytes = len(buffer_pcm) - MAX_BACKLOG_BYTES
            drop_chunks = int(np.ceil(overflow_bytes / CHUNK_BYTES)) # Convert overflow size from bytes to whole-chunk units
            drop_bytes = drop_chunks * CHUNK_BYTES

            del buffer_pcm[:drop_bytes] # Remove the oldest buffered bytes
            dropped_chunks_count += drop_chunks # Count the dropped chunks
            print(f'{drop_chunks}chunks dropped!') # Print the number of dropped chunks in this loop
        
        # Process chunk in the buffer
        while len(buffer_pcm) >= CHUNK_BYTES: 
            pcm_bytes = buffer_pcm[:CHUNK_BYTES] # Extract one chunk of PCM bytes

            x = np.frombuffer(pcm_bytes, dtype="<i4").astype(np.float32) # Convert raw little-endian int32 PCM bytes to float32 samples
            x /= (2**31) # Normalize int32 PCM values to [-1, 1]

            x = remove_dc_offset(x) # Remove DC offset before spectral analysis
            power_db = compute_powerspec_db(x, CHUNK_SIZE, WINDOW, FFT_SIZE) # Compute power spectrum in dB
            power_db = power_db[valid_freqs] # Keep only valid frequencies (upper 20 Hz)

            del buffer_pcm[:HOP_BYTES] # Remove the extracted chunk from the buffer
            chunk_count += 1 # Count the processed chunk

            now = time.time() 
            # Update the plot only at the specified frame rate
            if now - last_plot_time > PLOT_INTERVAL: 
                line.set_ydata(power_db) # Update the y-data of the line plot
                fig.canvas.restore_region(background) # Restore the saved background
                ax.draw_artist(line) # Draw the line artist on top of the background
                fig.canvas.blit(ax.bbox) # Update only the axes region for efficient real-time rendering
                fig.canvas.flush_events() # Process GUI events so the initial frame appears

                last_plot_time = now # Update the last plot time

            if chunk_count % 100 == 0: 
                print(f"chunk: {chunk_count}") # Print progress every 100 processed chunks

    elif channel.exit_status_ready(): 
        # Exit the loop when the remote process has finished and there is no more data left to read
        if not channel.recv_ready():
            break
    else:
        time.sleep(0.001) # Small sleep to avoid busy-waiting when no data is available

# Cleanup
err = stderr.read().decode("utf-8", errors="ignore") # Read stderr from the remote process
ssh.close() # Close SSH connection


print("---- arecord stderr ----")
print(err)
print(f"number of total chunks: {chunk_count}")
print(f"number of dropped chunks : {dropped_chunks_count}")

plt.ioff()